# 02 · Analysis — five questions a national procurement team would ask

**Scenario (fictional):** you are a data analyst for a catering chain that buys staple
food in all 34 provinces. Procurement wants to know *when* to stock up, *what* to lock
into fixed-price contracts, *where* prices run high, *who* in the supply chain to buy from,
and how heavy the food bill is relative to the local minimum wage.

Data: PIHPS Nasional (Bank Indonesia), 2018-01 → 2026-09, cleaned as described in
`01_data_quality.ipynb`. Every number below comes from the `mart_*` tables built by the SQL
in `sql/marts/`.

In [1]:
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

from pangan import plots
from pangan.config import SQL_DIR, WAREHOUSE_PATH
from pangan.quality import quality_report

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
con = duckdb.connect(str(WAREHOUSE_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


## Q1 · How much do prices rise around Lebaran, and when?

Index = national traditional-market price ÷ its average over H-60 … H-46 of the same year.
The size of the rise is the median of the yearly peaks (9 seasons, 2018–2026); timing is read
from the typical-season curve (week −1 = the last 7 days before Idul Fitri).

In [2]:
summary = q('''
SELECT c.commodity_name, s.median_peak_pct, s.start_rise_week, s.peak_week, s.recover_week,
       s.n_years
FROM mart_q1_summary s JOIN dim_commodity c USING (commodity_id)
ORDER BY s.median_peak_pct DESC
''')
summary.style.format({"median_peak_pct": "{:.1%}"}, na_rep="–")

,commodity_name,median_peak_pct,start_rise_week,peak_week,recover_week,n_years
0,Cabai Rawit Merah,17.7%,-5,-5,-4,9
1,Cabai Merah Besar,14.2%,-6,-4,-3,9
2,Daging Ayam Ras Segar,12.8%,-1,0,3,9
3,Cabai Merah Keriting,12.8%,0,0,1,9
4,Cabai Rawit Hijau,12.8%,–,-4,-3,9
5,Bawang Merah Ukuran Sedang,10.4%,1,3,–,9
6,Telur Ayam Ras Segar,8.1%,-1,-1,1,9
7,Bawang Putih Ukuran Sedang,6.0%,-1,-1,0,9
8,Daging Sapi Kualitas 1,5.4%,–,-1,0,9
9,Daging Sapi Kualitas 2,4.9%,–,-1,0,9


The yearly peaks vary a lot, and there are only 9 seasons — so the median rise gets a
bootstrap 95% confidence interval (10,000 resamples of the 9 seasons).

In [3]:
index = q('''
SELECT c.commodity_name, i.lebaran_year, i.t, i.price_index
FROM mart_q1_lebaran_index i JOIN dim_commodity c USING (commodity_id)
WHERE i.t >= -42
''')
yearly_peak = index.groupby(["commodity_name", "lebaran_year"]).price_index.max() - 1

rng = np.random.default_rng(42)


def bootstrap_ci(values: np.ndarray, n: int = 10_000) -> tuple[float, float]:
    samples = rng.choice(values, size=(n, len(values)), replace=True)
    return tuple(np.percentile(np.median(samples, axis=1), [2.5, 97.5]))


ci = (
    yearly_peak.groupby(level=0)
    .apply(lambda s: pd.Series(bootstrap_ci(s.to_numpy()), index=["ci_low", "ci_high"]))
    .unstack()
)
ci["median_peak"] = yearly_peak.groupby(level=0).median()
ci.sort_values("median_peak", ascending=False).head(10).style.format("{:.1%}")

,ci_low,ci_high,median_peak
commodity_name,,,
Cabai Rawit Merah,-4.5%,52.1%,17.7%
Cabai Merah Besar,8.1%,43.5%,14.2%
Daging Ayam Ras Segar,7.5%,17.5%,12.8%
Cabai Merah Keriting,3.8%,42.2%,12.8%
Cabai Rawit Hijau,-0.8%,24.4%,12.8%
Bawang Merah Ukuran Sedang,-1.9%,44.0%,10.4%
Telur Ayam Ras Segar,2.5%,15.0%,8.1%
Bawang Putih Ukuran Sedang,-2.7%,21.9%,6.0%
Daging Sapi Kualitas 1,2.8%,8.5%,5.4%


In [4]:
lebaran_plot = q('''
SELECT i.commodity_id, c.commodity_name, i.lebaran_year, i.t, i.price_index
FROM mart_q1_lebaran_index i JOIN dim_commodity c USING (commodity_id)
''')
for commodity_id, name in [("com_7", "lebaran_ayam"), ("com_16", "lebaran_cabai_rawit")]:
    plots.save_fig(fig := plots.plot_lebaran_index(lebaran_plot, commodity_id), name)
    display(fig)

<Figure size 800x450 with 1 Axes>

<Figure size 800x450 with 1 Axes>

## Q2 · Which commodities are least stable, and when were the shocks?

Volatility = standard deviation of week-on-week change of the national weekly price,
averaged over the years.

In [5]:
volatility = q('''
SELECT c.commodity_name, v.year, v.std_pct_1w
FROM mart_q2_volatility v JOIN dim_commodity c USING (commodity_id)
''')
plots.save_fig(fig := plots.plot_volatility(volatility), "volatility")
display(fig)
volatility.groupby("commodity_name").std_pct_1w.mean().sort_values(ascending=False).head(8).map(
    "{:.1%}".format
)

<Figure size 800x708 with 1 Axes>

commodity_name
Cabai Rawit Merah             8.5%
Cabai Merah Keriting          7.3%
Cabai Merah Besar             6.8%
Cabai Rawit Hijau             5.2%
Bawang Merah Ukuran Sedang    3.8%
Bawang Putih Ukuran Sedang    2.5%
Daging Ayam Ras Segar         2.2%
Telur Ayam Ras Segar          1.4%
Name: std_pct_1w, dtype: str

In [6]:
q('''
SELECT c.commodity_name, s.week, s.pct_4w
FROM mart_q2_top_shocks s JOIN dim_commodity c USING (commodity_id)
WHERE s.rank = 1
ORDER BY s.pct_4w DESC
''').style.format({"pct_4w": "{:+.1%}"})

,commodity_name,week,pct_4w
0,Cabai Rawit Merah,2021-12-13 00:00:00,+125.5%
1,Cabai Merah Keriting,2024-12-30 00:00:00,+70.4%
2,Cabai Rawit Hijau,2021-12-13 00:00:00,+63.1%
3,Cabai Merah Besar,2024-12-30 00:00:00,+62.3%
4,Bawang Putih Ukuran Sedang,2019-05-06 00:00:00,+62.1%
5,Bawang Merah Ukuran Sedang,2024-04-22 00:00:00,+54.9%
6,Minyak Goreng Kemasan Bermerk 2,2022-03-28 00:00:00,+35.1%
7,Minyak Goreng Kemasan Bermerk 1,2022-03-28 00:00:00,+31.2%
8,Daging Ayam Ras Segar,2020-05-25 00:00:00,+30.6%
9,Gula Pasir Lokal,2020-03-23 00:00:00,+25.0%


## Q3 · Where are prices high, and is the gap closing?

Index = a province's monthly average ÷ the median province that month (1.25 = 25% above).
Regions follow time zones: Barat = WIB, Tengah = WITA, Timur = WIT.

In [7]:
region = q('''
SELECT p.tz_region, c.commodity_name, AVG(q.index_vs_median) AS index_vs_median
FROM mart_q3_province_index q
JOIN dim_province p USING (province_id) JOIN dim_commodity c USING (commodity_id)
WHERE month >= DATE '2025-10-01'
GROUP BY ALL
''')
region.pivot(index="commodity_name", columns="tz_region", values="index_vs_median")[
    ["Barat", "Tengah", "Timur"]
].sort_values("Timur", ascending=False).style.format("{:.2f}")

tz_region,Barat,Tengah,Timur
commodity_name,,,
Cabai Rawit Merah,0.98,0.93,1.52
Cabai Merah Keriting,1.05,0.92,1.47
Bawang Merah Ukuran Sedang,0.94,1.01,1.39
Bawang Putih Ukuran Sedang,0.95,1.05,1.35
Cabai Merah Besar,1.04,0.94,1.31
Telur Ayam Ras Segar,0.97,1.02,1.26
Gula Pasir Kualitas Premium,1.00,1.02,1.24
Cabai Rawit Hijau,0.99,0.99,1.22
Daging Ayam Ras Segar,1.00,0.98,1.21


In [8]:
province_index = q('''
SELECT q.commodity_id, c.commodity_name, p.province_name, q.index_vs_median
FROM mart_q3_province_index q
JOIN dim_province p USING (province_id) JOIN dim_commodity c USING (commodity_id)
WHERE month >= DATE '2025-10-01'
''')
plots.save_fig(fig := plots.plot_province_index(province_index, "com_16"), "province_index_cabai")
display(fig)
overall = province_index.groupby("province_name").index_vs_median.mean().sort_values()
pd.concat([overall.tail(5), overall.head(5)]).map("{:.2f}".format)

<Figure size 800x1016 with 1 Axes>

province_name
Kalimantan Utara       1.14
Kalimantan Tengah      1.15
Papua Barat            1.17
Maluku Utara           1.23
Papua                  1.28
Sulawesi Selatan       0.86
Jawa Timur             0.89
Nusa Tenggara Barat    0.91
Sulawesi Barat         0.91
Jawa Tengah            0.93
Name: index_vs_median, dtype: str

Is the disparity narrowing? Spearman rank correlation between time and the monthly
coefficient of variation across provinces (negative ρ = gap closing).

In [9]:
disparity = q('''
SELECT c.commodity_name, d.month, d.cv
FROM mart_q3_disparity d JOIN dim_commodity c USING (commodity_id)
ORDER BY d.month
''')
trend = []
for name, g in disparity.groupby("commodity_name"):
    rho, p_value = spearmanr(g["month"].rank(), g["cv"])
    trend.append({
        "commodity_name": name,
        "cv_2018": g[g.month.dt.year == 2018].cv.mean(),
        "cv_last_12m": g.tail(12).cv.mean(),
        "spearman_rho": rho,
        "p_value": p_value,
    })
pd.DataFrame(trend).sort_values("spearman_rho").style.format(
    {"cv_2018": "{:.3f}", "cv_last_12m": "{:.3f}", "spearman_rho": "{:+.2f}", "p_value": "{:.1g}"}
)

,commodity_name,cv_2018,cv_last_12m,spearman_rho,p_value
20,Telur Ayam Ras Segar,0.160,0.118,-0.77,1e-21
4,Beras Kualitas Medium I,0.110,0.085,-0.74,4e-19
2,Beras Kualitas Bawah I,0.110,0.095,-0.70,1e-16
15,Gula Pasir Kualitas Premium,0.112,0.096,-0.51,3e-08
16,Gula Pasir Lokal,0.069,0.057,-0.39,4e-05
1,Bawang Putih Ukuran Sedang,0.183,0.156,-0.37,8e-05
17,Minyak Goreng Curah,0.094,0.083,-0.28,0.004
7,Beras Kualitas Super II,0.097,0.092,-0.28,0.004
5,Beras Kualitas Medium II,0.111,0.099,-0.27,0.005
3,Beras Kualitas Bawah II,0.106,0.100,-0.25,0.01


## Q4 · Where does the price grow along the supply chain?

National series, average of the last 12 months. *Pasar Modern* is an alternative retail
channel rather than a later stage, so the last step is the gap between two ways to buy retail.

In [10]:
markup = q('''
SELECT c.commodity_name, m.from_order, m.from_level, m.to_level, AVG(m.markup_pct) AS markup_pct
FROM mart_q4_markup m JOIN dim_commodity c USING (commodity_id)
WHERE month >= DATE '2025-10-01'
GROUP BY ALL
''')
chosen = ["Cabai Merah Besar", "Cabai Rawit Merah", "Bawang Merah Ukuran Sedang",
          "Bawang Putih Ukuran Sedang", "Daging Ayam Ras Segar", "Telur Ayam Ras Segar",
          "Beras Kualitas Medium I", "Minyak Goreng Curah"]
plots.save_fig(fig := plots.plot_markup(markup[markup.commodity_name.isin(chosen)]
                                         .sort_values("from_order")), "markup")
display(fig)
wide = markup.pivot_table(index="commodity_name", columns="from_order", values="markup_pct")
wide.columns = ["produsen→pedagang_besar", "pedagang_besar→tradisional", "tradisional→modern"]
wide["produsen→tradisional"] = (1 + wide.iloc[:, 0]) * (1 + wide.iloc[:, 1]) - 1
wide.sort_values("produsen→tradisional", ascending=False).style.format("{:+.1%}")

<Figure size 800x456 with 1 Axes>

,produsen→pedagang_besar,pedagang_besar→tradisional,tradisional→modern,produsen→tradisional
commodity_name,,,,
Bawang Putih Ukuran Sedang,+48.8%,+27.3%,+28.9%,+89.5%
Cabai Merah Keriting,+24.3%,+28.9%,+51.1%,+60.3%
Cabai Rawit Hijau,+32.9%,+16.5%,+61.8%,+54.8%
Cabai Rawit Merah,+19.0%,+24.6%,+45.1%,+48.3%
Bawang Merah Ukuran Sedang,+16.7%,+25.5%,+41.9%,+46.4%
Cabai Merah Besar,+16.2%,+25.6%,+72.1%,+45.9%
Daging Ayam Ras Segar,+15.7%,+19.3%,+18.0%,+38.1%
Gula Pasir Kualitas Premium,+15.0%,+8.8%,+4.0%,+25.1%
Beras Kualitas Super II,+14.6%,+5.5%,+1.8%,+20.9%


## Q5 · How heavy is the food bill relative to the minimum wage?

Monthly basket for a 4-person household (BPS Susenas per-capita quantities) priced at each
province's traditional-market average, divided by the provincial minimum wage (UMP).

In [11]:
afford = q('''
SELECT p.province_name, a.year, a.month, a.basket_cost_idr, a.ump_idr, a.share_of_ump
FROM mart_q5_affordability a JOIN dim_province p USING (province_id)
''')
if afford.empty:
    print("UMP / basket reference data from BPS not loaded yet - see docs/DATA_SOURCES.md.")
else:
    latest = afford[afford.year == afford.year.max()]
    plots.save_fig(fig := plots.plot_affordability(latest), "affordability")
    display(fig)
    display(latest.groupby("province_name").share_of_ump.mean().sort_values(ascending=False)
            .map("{:.1%}".format).head(10))

UMP / basket reference data from BPS not loaded yet - see docs/DATA_SOURCES.md.
